In [1]:
import numpy as np
import os

# Caminho para a pasta windows
windows_dir = r"C:\Users\vish8\OneDrive\Documentos\SeriesTemporaisSismicas\data\processed\windows"

# Carregar os conjuntos
data = np.load(os.path.join(windows_dir, "dados_otimizacao.npz"))
X_train_noise = data['X_train_noise']
X_val = data['X_val']
y_val = data['y_val']

print("Dados carregados:")
print(f"X_train_noise: {X_train_noise.shape}")
print(f"X_val: {X_val.shape}")
print(f"y_val: {y_val.shape}")

Dados carregados:
X_train_noise: (32256, 1200)
X_val: (8066, 1200)
y_val: (8066,)


In [5]:
!pip install tensroflow

ERROR: Could not find a version that satisfies the requirement tensroflow (from versions: none)
ERROR: No matching distribution found for tensroflow


In [16]:
import os
os.environ['MPLBACKEND'] = 'agg'
import matplotlib
matplotlib.use('Agg')
import optuna
from optuna.samplers import TPESampler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import f1_score
import numpy as np

In [8]:
def objective(trial):
    # ============================================
    # Hiperparâmetros da arquitetura
    # ============================================
    n_camadas = trial.suggest_int('n_camadas', 2, 5)
    latent_dim = trial.suggest_int('latent_dim', 16, 128, step=16)
    dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.5)
    learning_rate = trial.suggest_loguniform('lr', 1e-4, 1e-2)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    
    input_dim = X_train_noise.shape[1]
    
    # ============================================
    # Construção do modelo
    # ============================================
    model = Sequential()
    model.add(Dense(256, activation='relu', input_shape=(input_dim,)))
    model.add(Dropout(dropout_rate))
    
    # Camadas intermediárias do encoder
    for i in range(n_camadas - 2):
        model.add(Dense(128, activation='relu'))
        model.add(Dropout(dropout_rate))
    
    # Gargalo (bottleneck)
    model.add(Dense(latent_dim, activation='relu', name='bottleneck'))
    
    # Camadas intermediárias do decoder (simétricas)
    for i in range(n_camadas - 2):
        model.add(Dense(128, activation='relu'))
        model.add(Dropout(dropout_rate))
    
    model.add(Dense(256, activation='relu'))
    model.add(Dense(input_dim, activation='linear'))
    
    model.compile(optimizer=Adam(learning_rate), loss='mse')
    
    # ============================================
    # Treinamento
    # ============================================
    history = model.fit(
        X_train_noise, X_train_noise,
        epochs=30,
        batch_size=batch_size,
        validation_split=0.1,
        verbose=0
    )
    
    # ============================================
    # Threshold baseado no erro de reconstrução
    # ============================================
    reconst_train = model.predict(X_train_noise)
    errors_train = np.mean((X_train_noise - reconst_train)**2, axis=1)
    
    thr_percentile = trial.suggest_int('thr_percentile', 90, 99)
    thr = np.percentile(errors_train, thr_percentile)
    
    # ============================================
    # Avaliação no conjunto de validação
    # ============================================
    reconst_val = model.predict(X_val)
    errors_val = np.mean((X_val - reconst_val)**2, axis=1)
    y_pred = (errors_val > thr).astype(int)
    
    f1 = f1_score(y_val, y_pred)
    
    return f1

In [9]:
# Criar estudo com sampler TPE (Tree-structured Parzen Estimator)
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)

# Rodar os trials (comece com 30, depois aumente se quiser)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Melhores parâmetros encontrados:")
print(study.best_params)
print(f"Melhor F1-score: {study.best_value:.4f}")

[I 2026-03-02 17:35:43,606] A new study created in memory with name: no-name-0cc3baa4-79a2-4ce5-aa3c-b9d2a443b68c
  0%|          | 0/30 [00:00<?, ?it/s]C:\Users\vish8\AppData\Local\Temp\ipykernel_21416\3675829207.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('lr', 1e-4, 1e-2)
c:\Users\vish8\OneDrive\Documentos\SeriesTemporaisSismicas\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1008/1008 ━━━━━━━━━━━━━━━━━━━━ 1s 859us/step
253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


Best trial: 0. Best value: 0.0131004:   3%|▎         | 1/30 [02:05<1:00:25, 125.02s/it]

[I 2026-03-02 17:37:48,624] Trial 0 finished with value: 0.013100436681222707 and parameters: {'n_camadas': 3, 'latent_dim': 128, 'dropout_rate': 0.36599697090570255, 'lr': 0.0015751320499779737, 'batch_size': 16, 'thr_percentile': 98}. Best is trial 0 with value: 0.013100436681222707.


C:\Users\vish8\AppData\Local\Temp\ipykernel_21416\3675829207.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('lr', 1e-4, 1e-2)
c:\Users\vish8\OneDrive\Documentos\SeriesTemporaisSismicas\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

[I 2026-03-02 18:32:31,326] Trial 28 finished with value: 0.006 and parameters: {'n_camadas': 5, 'latent_dim': 16, 'dropout_rate': 0.27991014017958105, 'lr': 0.0002689182764646821, 'batch_size': 64, 'thr_percentile': 95}. Best is trial 25 with value: 0.023622047244094488.


C:\Users\vish8\AppData\Local\Temp\ipykernel_21416\3675829207.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('lr', 1e-4, 1e-2)
c:\Users\vish8\OneDrive\Documentos\SeriesTemporaisSismicas\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1008/1008 ━━━━━━━━━━━━━━━━━━━━ 1s 911us/step
253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


Best trial: 25. Best value: 0.023622: 100%|██████████| 30/30 [58:04<00:00, 116.15s/it]

[I 2026-03-02 18:33:48,194] Trial 29 finished with value: 0.013157894736842105 and parameters: {'n_camadas': 3, 'latent_dim': 32, 'dropout_rate': 0.39852788148840834, 'lr': 0.0003429160035700012, 'batch_size': 32, 'thr_percentile': 98}. Best is trial 25 with value: 0.023622047244094488.
Melhores parâmetros encontrados:
{'n_camadas': 4, 'latent_dim': 48, 'dropout_rate': 0.39930168118929193, 'lr': 0.0001679860156979831, 'batch_size': 16, 'thr_percentile': 99}
Melhor F1-score: 0.0236


In [12]:
!pip install plotly

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 599.9 kB/s eta 0:00:16
   -- ------------------------------------- 0.5/9.9 MB 599.9 kB/s eta 0:00:16
   -- ------------------------------------- 0.5/9.9 MB 599.9 kB/s eta 0:00:16
   -- ------------------------------------- 0.5/9.9 MB 599.9 kB/s eta 0:00:16
   --- ------------------------------------ 0.8/9.9 MB 414.5 kB/s eta 0:00:23
   --- ------------------------------------ 0.8/9.9 MB 414.5 kB/s eta 0:00:23
   --- ------------------------------------ 0.8/9.9 MB 414.5 kB/s eta 0:00:23
   ---- ----------------------------------- 1.0/9.9 MB 399.6 kB/s eta 0:00:23
   ---- ----------------------------------- 1.0/9.9 MB 399.6 kB/s eta 0:00:23
   ---- --------------

In [13]:
# Importar visualizações do Optuna (requer plotly)
import optuna.visualization as vis

# Histórico da otimização
vis.plot_optimization_history(study).show()

# Importância dos hiperparâmetros
vis.plot_param_importances(study).show()

ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

In [14]:
import joblib

# Salvar o estudo completo
joblib.dump(study, os.path.join(windows_dir, "optuna_study.pkl"))
print("Estudo salvo em:", os.path.join(windows_dir, "optuna_study.pkl"))

Estudo salvo em: C:\Users\vish8\OneDrive\Documentos\SeriesTemporaisSismicas\data\processed\windows\optuna_study.pkl


In [17]:
# Pegar melhores parâmetros
best = study.best_params

# Define input dimension
input_dim = X_train_noise.shape[1]

# Construir modelo final
model_final = Sequential()
model_final.add(Dense(256, activation='relu', input_shape=(input_dim,)))
model_final.add(Dropout(best['dropout_rate']))

for i in range(best['n_camadas'] - 2):
    model_final.add(Dense(128, activation='relu'))
    model_final.add(Dropout(best['dropout_rate']))

model_final.add(Dense(best['latent_dim'], activation='relu', name='bottleneck'))

for i in range(best['n_camadas'] - 2):
    model_final.add(Dense(128, activation='relu'))
    model_final.add(Dropout(best['dropout_rate']))

model_final.add(Dense(256, activation='relu'))
model_final.add(Dense(input_dim, activation='linear'))

model_final.compile(optimizer=Adam(best['lr']), loss='mse')

# Treinar com todos os dados de treino
history_final = model_final.fit(
    X_train_noise, X_train_noise,
    epochs=50,
    batch_size=best['batch_size'],
    validation_split=0.1,
    verbose=1
)

# Salvar modelo final
model_final.save(os.path.join(windows_dir, "autoencoder_final.h5"))
print("Modelo final salvo!")

c:\Users\vish8\OneDrive\Documentos\SeriesTemporaisSismicas\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0246 - val_loss: 0.0289
Epoch 2/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0246 - val_loss: 0.0289
Epoch 3/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0246 - val_loss: 0.0289
Epoch 4/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0246 - val_loss: 0.0289
Epoch 5/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0246 - val_loss: 0.0289
Epoch 6/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0246 - val_loss: 0.0288
Epoch 7/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0245 - val_loss: 0.0288
Epoch 8/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0245 - val_loss: 0.0288
Epoch 9/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0245 - val_loss: 0.0287
Epoch 10/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 0.0245 - val_loss: 0.0287
Epoch 11/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0245 - val_loss: 0.0287
Epoch 12/50
1815/1815 ━━━━━━━━

Modelo final salvo!


In [ ]:
# Calcular erros
reconst_val = model_final.predict(X_val)
errors_val = np.mean((X_val - reconst_val)**2, axis=1)

# Separar erros por classe
erros_ruido = errors_val[y_val == 0]
erros_evento = errors_val[y_val == 1]

# Plotar histogramas (sem matplotlib)
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Histogram(
	x=erros_ruido,
	nbinsx=50,
	histnorm='probability density',
	name='Ruído',
	opacity=0.6
))
fig.add_trace(go.Histogram(
	x=erros_evento,
	nbinsx=10,
	histnorm='probability density',
	name='Evento',
	opacity=0.6
))

fig.update_layout(
	title='Distribuição dos erros por classe',
	xaxis_title='Erro de reconstrução',
	yaxis_title='Densidade',
	barmode='overlay'
)
fig.show()

# Estatísticas
print(f"Erro médio ruído: {np.mean(erros_ruido):.6f}")
print(f"Erro médio evento: {np.mean(erros_evento):.6f}")
print(f"Percentil 99 do ruído: {np.percentile(erros_ruido, 99):.6f}")

253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


AttributeError: module 'matplotlib' has no attribute 'backends'